## Data Visualization 
This part is dedicated to the analysis of our data in order to chose the best fitting model, and its parametters 

In [1]:
include("../Data_Processing.jl")
using StatsBase,StatsPlots

In [2]:
filepath = joinpath(pwd(), "..", "..", "Data", "student-mat.csv")
df = loader(filepath)
size(df)

(395, 33)

### Entropy calculations and data visualisation

We start by calculating the main entropy of our dataset

In [3]:
Pass = subset(df,:G3 => ByRow(x->x>=10))
NoPass = subset(df,:G3 =>ByRow(x->x<10))
[size(Pass),size(NoPass)]

2-element Vector{Tuple{Int64, Int64}}:
 (265, 33)
 (130, 33)

In [4]:
H=-(length(Pass.G3)/length(df.G3))*log2(length(Pass.G3)/length(df.G3))-(length(NoPass.G3)/length(df.G3))*log2(length(NoPass.G3)/length(df.G3))

0.9140185106642175

We create a row that will allow us to classify our data to differenciate between the elements who passed and who failed

In [5]:
df.Pass = df.G3.>=10
S=nrow(df)

395

#### Definition of a entropy calculation function

In [6]:
function E_Calc(Sv::DataFrame)
    PL=count(Sv.Pass)
    NPL=nrow(Sv)- PL
    if (PL==0 || NPL == 0 )
        return 0
    else
        return E=  -((PL/(PL+NPL))*log2(PL/(PL+NPL)))-((NPL/(PL+NPL))*log2(NPL/(PL+NPL)))
    end
end

E_Calc (generic function with 1 method)

Checking the validity of out function by comparing it to our main entropy

In [21]:

println(H)
E_Calc(df)

0.9140185106642175


0.9140185106642175

#### Definition of a Information Gain calculation function

In [7]:
function IG_CalcG(H :: Float64 ,Sv :: DataFrame,S::Int64)
        return (H - ((nrow(Sv)/S)*E_Calc(Sv)))
end

IG_CalcG (generic function with 1 method)

#### Testing the information gain for each study hour

In [9]:
hours4 = subset(df, :studytime => ByRow(x->x==4))
hours3 = subset(df, :studytime => ByRow(x->x==3))
hours2 = subset(df, :studytime => ByRow(x->x==2))
hours1 = subset(df, :studytime => ByRow(x->x==1))
[nrow(hours1),nrow(hours2),nrow(hours3),nrow(hours4)]

4-element Vector{Int64}:
 105
 198
  65
  27

In [22]:
Hours = [hours1,hours2,hours3,hours4]
size(Hours)

(4,)

In [ ]:
IG = H
for Sv in Hours
    IG=IG_CalcG(IG,Sv,S)
    print(IG, "\n")
end
IG

0.6651614685840943
0.19538222911671443
0.06289330383771613
0.006458072935784036


0.006458072935784036

Very low information gain, not useful for our main branching.

### Testing the information gain for the first periode grades

In [14]:
#Grade 1 over average
G1OA = subset(df,:G1 => ByRow(x->x>=10))
#Grades 1 under average
G1UA = subset(df,:G1 => ByRow(x->x<10))
IG=H 
Grades1=[G1OA,G1UA]
for Sv in Grades1
    IG = IG_CalcG(IG,Sv,S)
    println(IG)
end
IG

0.6080452572187635
0.3067867388082951


0.3067867388082951

Meaningful Information gain

In [20]:
#Grade 1 over average
G2OA = subset(df,:G2 => ByRow(x->x>=10))
#Grades 1 under average
G2UA = subset(df,:G2 => ByRow(x->x<10))

Grades2=[G2OA,G2UA]
IG = H 
for Sv in Grades2
    IG = IG_CalcG(IG,Sv,S)
    println(IG)
end
IG

0.7848181282502139
0.5465261525628654


0.5465261525628654

Important Information Gain